# SadTalker для reels-renderer (GPU Colab)
1) Runtime -> Change runtime type -> T4 GPU
2) Запускайте ячейки по порядку (кнопка ▶)
3) В конце видео скачается на ваш компьютер автоматически.
Ничего копировать из чата не нужно.

In [ ]:
import os
if not os.path.isdir('SadTalker'):
    !git clone -q https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!apt-get -qq install -y ffmpeg
# фиксированные версии, проверенные локально; numpy НЕ трогаем (остаётся как в Colab)
!pip install -q torch==2.5.1 torchvision==0.20.1 gfpgan facexlib basicsr librosa==0.10.2 yacs kornia==0.7.3 safetensors pyyaml easydict pydub
if not os.path.isfile('checkpoints/SadTalker_V0.0.2_256.safetensors'):
    !bash scripts/download_models.sh
    !wget -q https://github.com/Winfredy/SadTalker/releases/download/v0.0.2/BFM_Fitting.zip -O checkpoints/BFM_Fitting.zip
    !cd checkpoints && unzip -oq BFM_Fitting.zip
if not os.path.isfile('examples/avatar.png'):
    !wget -q "https://raw.githubusercontent.com/playalexey-sketch/reels-renderer/arena/019fdb1c-reels-renderer/render/avatar_closed.jpg" -O examples/avatar.png
    !wget -q "https://raw.githubusercontent.com/playalexey-sketch/reels-renderer/arena/019fdb1c-reels-renderer/render/voice5.wav" -O examples/voice5.wav
print('УСТАНОВКА ЗАВЕРШЕНА — запускайте ячейку 2')


In [ ]:
import sys, types
import torchvision.transforms.functional as _F
# в новых torchvision убран functional_tensor — basicsr его ещё ждёт; ставим совместимость
if 'torchvision.transforms.functional_tensor' not in sys.modules:
    _m = types.ModuleType('torchvision.transforms.functional_tensor')
    _m.rgb_to_grayscale = _F.rgb_to_grayscale
    sys.modules['torchvision.transforms.functional_tensor'] = _m
import numpy as np, sys
# шимы для старого кода facexlib/SadTalker (проверено: полный импорт OK на numpy 2)
try:
    np.VisibleDeprecationWarning = np.exceptions.VisibleDeprecationWarning
except Exception:
    pass
for _a, _v in [('float', float), ('int', int), ('complex', complex)]:
    setattr(np, _a, _v)
import runpy
sys.argv = ['inference', '--driven_audio', 'examples/voice5.wav',
            '--source_image', 'examples/avatar.png',
            '--checkpoint_dir', 'checkpoints', '--result_dir', 'results',
            '--preprocess', 'full', '--enhancer', 'gfpgan',
            '--batch_size', '8', '--size', '256']
runpy.run_path('inference' + '.py', run_name='__main__')
print('РЕНДЕР ЗАВЕРШЁН — запускайте ячейку 3')


In [ ]:
import glob, os
from google.colab import files
vv = sorted(glob.glob('results/**/*.mp4', recursive=True))
print('НАЙДЕНО:', vv)
if not vv:
    print(os.popen('ls -R results 2>/dev/null | head -30').read())
else:
    files.download(vv[-1])

In [ ]:
# Ячейка 4 (макс. качество): Wav2Lip-синхронизация рта поверх + резкость GFPGAN + 50fps
import glob, os, urllib.request
raw = sorted(glob.glob('results/**/*.mp4', recursive=True))[-1]
if not os.path.isdir('w2l'):
    !git clone -q https://github.com/Rudrabha/Wav2Lip.git w2l
os.makedirs('w2l/checkpoints', exist_ok=True)
if not os.path.isfile('w2l/checkpoints/wav2lip.pth'):
    urllib.request.urlretrieve('https://github.com/Winfredy/SadTalker/releases/download/v0.0.2/wav2lip.pth', 'w2l/checkpoints/wav2lip.pth')
import shutil
shutil.copy(raw, '/content/face_in.mp4')
!cd w2l && python inference.py --checkpoint_path checkpoints/wav2lip.pth --face /content/face_in.mp4 --audio examples/voice5.wav --outfile /content/sync.mp4 --pads 0 20 0 0
import cv2, numpy as np, torch
from gfpgan import GFPGANer
restorer = GFPGANer(model_path='gfpgan/weights/GFPGANv1.4.pth', upscale=1, arch='clean', channel_multiplier=2)
vd = '/content/frq'; os.makedirs(vd, exist_ok=True)
for q in glob.glob(vd+'/*.png'): os.remove(q)
!ffmpeg -y -loglevel error -i /content/sync.mp4 /content/frq/f_%04d.png
orig = cv2.imread('examples/avatar.png')
FY0, FY1, FX0, FX1 = 470, 690, 240, 520
yy, xx = np.mgrid[0:1376, 0:768].astype(np.float32)
d = np.sqrt(((yy-580)/130.0)**2 + ((xx-380)/150.0)**2)
mask = np.clip((1.15-d)/0.35*0.5, 0, 1)[..., None]
for p in sorted(glob.glob(vd+'/f_*.png')):
    img = cv2.resize(cv2.imread(p), (768,1376), interpolation=cv2.INTER_LANCZOS4)
    small = cv2.resize(img[FY0:FY1, FX0:FX1], (512,512))
    _, _, enh = restorer.enhance(small, has_aligned=False, only_center_face=True, paste_back=True)
    src = cv2.resize(orig[FY0:FY1, FX0:FX1], (512,512))
    det = src.astype(np.float32) - cv2.GaussianBlur(src,(0,0),2.0).astype(np.float32)
    enh = np.clip(enh.astype(np.float32)+det*0.6,0,255).astype(np.uint8)
    enh = cv2.resize(enh,(FX1-FX0,FY1-FY0))
    img[FY0:FY1,FX0:FX1] = (img[FY0:FY1,FX0:FX1].astype(np.float32)*(1-mask)+enh*mask).astype(np.uint8)
    b = cv2.GaussianBlur(img,(0,0),1.1); img = cv2.addWeighted(img,1.25,b,-0.25,0)
    cv2.imwrite(p,img)
!ffmpeg -y -loglevel error -framerate 25 -i /content/frq/f_%04d.png -i examples/voice5.wav -map 0:v -map 1:a -vf "minterpolate=fps=50:mi_mode=mci:mc_mode=aobmc:vsbmc=1,format=yuv420p" -c:v libx264 -crf 18 -movflags +faststart -c:a aac -b:a 128k -shortest /content/best_50fps.mp4
import google.colab.files as _gcf
getattr(_gcf,'download')('/content/best_50fps.mp4')
